In [ ]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [1]:
import torch
import pandas as pd
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import langid
import numpy as np
from torch import argmax

In [ ]:
MODEL_NAME = "matous-volf/political-leaning-deberta-large"

TEXT_COLUMN = "text"
LABEL_COLUMN = "annotation"

In [ ]:
raw_df = pd.read_csv("sample_annotated.csv")
langid.set_languages(['en', 'fr'])
raw_df["language"] = raw_df["text"].apply(lambda x: langid.classify(x)[0])
df = raw_df[raw_df['annotation']!='Unclassifiable']
df_en = df[df['language']=='en']
df_fr = df[df['language']=='fr']

texts = df_fr[TEXT_COLUMN].astype(str).tolist()
labels = df_fr[LABEL_COLUMN].astype(str).str.upper().tolist()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-large', use_fast=False)

model = AutoModelForSequenceClassification.from_pretrained(
    "matous-volf/political-leaning-deberta-large"
    ).to(device)

# launch/POLITICS, matous-volf/political-leaning-politics (68-48)


In [ ]:
model.eval()

max_length = model.config.max_position_embeddings

In [ ]:
BATCH_SIZE = 64
N_CHUNKS = (len(texts)//16)+1
parts = np.array_split(texts, N_CHUNKS)
print(f"Chunk sizes: {[len(p) for p in parts]}")

In [ ]:
# y_true = [0 if l=='LEFT' else 1 for l in labels]
# y_pred = []
with torch.no_grad():
    for i, chunk in enumerate(parts):
        encodings = tokenizer(
            chunk.tolist(),
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )

        encodings = {k: v.to(device) for k, v in encodings.items()}

        outputs = model(**encodings)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=-1).cpu().numpy()
        preds = ['Left' if l==0 else 'Right' for l in preds]
        
        df_temp = pd.DataFrame(chunk, columns=['text'])
        df_temp['left_right'] = preds
        df_temp.to_csv(f'output/Eng-left-right_{i}.csv', index=False)
